In [ ]:
# Imports
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints
openai = OpenAI()

# And OpenAI allows you to change the base_url
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

In [ ]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

In [ ]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

In [ ]:
response = openai.chat.completions.create(model="gpt-5-mini", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

In [ ]:
hard = """
On a bookshelf, two volumes of Harry Potter stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=hard_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))


In [ ]:
response = openai.chat.completions.create(model="gpt-5", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

In [ ]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5", messages=dilemma)
display(Markdown(response.choices[0].message.content))

In [ ]:
requests.get("http://localhost:11434/").content

In [ ]:
response = ollama.chat.completions.create(model="qwen3:8b", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

In [ ]:
# Gemini client library
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemma-4-26b-a4b-it", contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
print(response.text)

In [ ]:
# Using OpenRouter which is a proxy for multiple LLM providers 
response = openrouter.chat.completions.create(model="z-ai/glm-4.5", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

In [ ]:
# Using LangChain to connect to OpenAI. LangChain is a framework for building LLM applications.
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini")
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

In [ ]:
# Using Litellm to connect to OpenAI. Litellm is a lightweight API wrapper for multiple LLM providers.
from litellm import completion
response = completion(model="openai/gpt-4.1", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

In [ ]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

In [ ]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

In [ ]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [ ]:
response = completion(model="openai/gpt-5-nano", messages=question)
display(Markdown(response.choices[0].message.content))

In [ ]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

In [ ]:
question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet

In [ ]:
response = completion(model="openai/gpt-5-nano", messages=question)
display(Markdown(response.choices[0].message.content))

In [ ]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

In [ ]:
# Prompt Caching: It's a technique to cache the prompt and reuse it for future requests.
response = completion(model="openai/gpt-5-nano", messages=question)
display(Markdown(response.choices[0].message.content))

In [ ]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

In [ ]:
# We will now have a conversation with two chatbots both using GPT-5 mini.
model = "gpt-5-mini"

argumentative_system = """
You are a chatbot who is very argumentative.
You disagree with anything in the conversation and challenge everything in a snarky way.
"""

polite_system = """
You are a very polite and courteous chatbot.
You try to agree with everything the other person says or find common ground.
If the other person is argumentative, you remain calm and keep the conversation friendly.
"""

argumentative_messages = ["Hi there"]
polite_messages = ["Hi"]

In [ ]:
# Define a function to call the argumentative chatbot
def call_argumentative():
    messages = [
        {"role": "system", "content": argumentative_system}
    ]
    for argumentative, polite in zip(argumentative_messages, polite_messages):
        messages.append({"role": "assistant", "content": argumentative})
        messages.append({"role": "user", "content": polite})
    response = openai.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

In [ ]:
# Define a function to call the polite chatbot
def call_polite():
    messages = [
        {"role": "system", "content": polite_system}
    ]
    for argumentative, polite in zip(argumentative_messages, polite_messages):
        messages.append({"role": "user", "content": argumentative})
        messages.append({"role": "assistant", "content": polite})
    messages.append(
        {"role": "user", "content": argumentative_messages[-1]}
    )
    response = openai.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

In [ ]:
# Initialize the messages for the chatbots
argumentative_messages = ["Hi there"]
polite_messages = ["Hi"]

display(Markdown(f"### Argumentative GPT\n{argumentative_messages[0]}\n"))
display(Markdown(f"### Polite GPT\n{polite_messages[0]}\n"))

# Have a conversation with the chatbots
for i in range(5):
    argumentative_next = call_argumentative()
    display(Markdown(f"### Argumentative GPT\n{argumentative_next}\n"))
    argumentative_messages.append(argumentative_next)

    polite_next = call_polite()
    display(Markdown(f"### Polite GPT\n{polite_next}\n"))
    polite_messages.append(polite_next)